# 14 · Leer y escribir archivos

Los programas guardan información en archivos para que persista después de cerrarse.

## `open()` y modos

| Modo | Significado |
|------|-------------|
| `"r"` | leer (por defecto). Error si no existe |
| `"w"` | escribir. **Borra** el contenido previo |
| `"a"` | agregar al final |
| `"r+"` | leer y escribir |

Siempre usa `with`: cierra el archivo automáticamente, incluso si hay errores. Indica `encoding="utf-8"` para evitar problemas con tildes y ñ.

## Escribir

In [ ]:
with open("notas.txt", "w", encoding="utf-8") as f:
    f.write("Primera línea\n")      # write NO agrega salto de línea
    f.write("Segunda línea\n")
    f.writelines(["Tercera\n", "Cuarta\n"])
print("Archivo creado")

## Leer

In [ ]:
with open("notas.txt", encoding="utf-8") as f:
    contenido = f.read()      # todo como una cadena
print(contenido)

In [ ]:
with open("notas.txt", encoding="utf-8") as f:
    lineas = f.readlines()    # lista de líneas (con '\n')
print(lineas)

In [ ]:
# Forma más común: recorrer línea por línea
with open("notas.txt", encoding="utf-8") as f:
    for numero, linea in enumerate(f, start=1):
        print(numero, linea.strip())   # strip() quita el '\n'

## Agregar al final

In [ ]:
with open("notas.txt", "a", encoding="utf-8") as f:
    f.write("Línea agregada\n")

with open("notas.txt", encoding="utf-8") as f:
    print(f.read())

## ¿Qué pasa si el archivo no existe?

Se puede manejar el error con `try` / `except`.

In [ ]:
try:
    with open("no_existe.txt") as f:
        print(f.read())
except FileNotFoundError:
    print("El archivo no existe")

También se puede preguntar antes:

In [ ]:
from pathlib import Path

print(Path("notas.txt").exists())
print(Path("no_existe.txt").exists())

## Ejemplo: guardar y leer datos estructurados

### CSV a mano

In [ ]:
productos = [("Lápiz", 5), ("Cuaderno", 25), ("Borrador", 3)]

with open("productos.csv", "w", encoding="utf-8") as f:
    f.write("nombre,precio\n")
    for nombre, precio in productos:
        f.write(f"{nombre},{precio}\n")

total = 0
with open("productos.csv", encoding="utf-8") as f:
    next(f)   # saltar el encabezado
    for linea in f:
        nombre, precio = linea.strip().split(",")
        total += int(precio)
        print(nombre, precio)
print("Total:", total)

### Con el módulo `csv` (maneja comas y comillas)

In [ ]:
import csv

with open("productos.csv", encoding="utf-8", newline="") as f:
    for fila in csv.DictReader(f):
        print(fila)      # cada fila es un diccionario

### JSON: guardar diccionarios y listas

In [ ]:
import json

datos = {"usuario": "ana", "puntos": [10, 20, 30], "activo": True}

with open("datos.json", "w", encoding="utf-8") as f:
    json.dump(datos, f, indent=2, ensure_ascii=False)

with open("datos.json", encoding="utf-8") as f:
    leidos = json.load(f)
print(leidos, leidos == datos)

## Ejemplo: contar palabras de un archivo

In [ ]:
with open("texto.txt", "w", encoding="utf-8") as f:
    f.write("la casa es grande\nla casa es azul\n")

conteo = {}
with open("texto.txt", encoding="utf-8") as f:
    for linea in f:
        for palabra in linea.split():
            conteo[palabra] = conteo.get(palabra, 0) + 1
print(conteo)

## 🎛️ Bloc de notas

Escribe y guarda en un archivo; luego lee su contenido.

In [ ]:
import ipywidgets as widgets

area = widgets.Textarea(placeholder="Escribe algo...", layout=widgets.Layout(width="400px", height="80px"))
guardar = widgets.Button(description="Guardar (modo 'a')", button_style="success")
borrar = widgets.Button(description="Vaciar (modo 'w')", button_style="warning")
out = widgets.Output()

def mostrar():
    out.clear_output()
    with out:
        with open("bloc.txt", encoding="utf-8") as f:
            print("Contenido de bloc.txt:\n" + f.read())

def on_guardar(_):
    with open("bloc.txt", "a", encoding="utf-8") as f:
        f.write(area.value + "\n")
    area.value = ""
    mostrar()

def on_borrar(_):
    open("bloc.txt", "w").close()
    mostrar()

guardar.on_click(on_guardar)
borrar.on_click(on_borrar)
open("bloc.txt", "a").close()   # crea el archivo si no existe
display(area, widgets.HBox([guardar, borrar]), out)
mostrar()

## Resumen

- `with open(ruta, modo, encoding="utf-8") as f:`
- `f.read()`, `f.readlines()`, `for linea in f:`, `f.write()`.
- `"w"` sobrescribe, `"a"` agrega.
- `csv` y `json` para datos estructurados.